In [ ]:
# Cell 1 - Install dependencies
import sys
!{sys.executable} -m pip install chromadb sentence-transformers tqdm kagglehub -q

In [ ]:
# Cell 2 - Detect environment and set paths
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Local'}")

if ON_KAGGLE:
    QUERIES_PATH = Path("/kaggle/input/datasets/yanbochen928/fairsearch-qbio-queries-yb/queries_v2_draft.json")
    CHROMA_SRC   = Path("/kaggle/input/datasets/yanbochen928/fairsearch-qbio-chromadb/chroma_db")
    CHROMA_PATH  = Path("/kaggle/working/chroma_db")
    OUT_PATH     = Path("/kaggle/working/retrieval_results_v2_q051_q150.json")
else:
    REPO_ROOT = next(
        c.resolve() for c in [Path("."), Path("..")]
        if (c / "queries" / "queries.json").exists()
    )
    QUERIES_PATH = REPO_ROOT / "queries" / "queries.json"
    CHROMA_SRC   = None
    CHROMA_PATH  = REPO_ROOT / "data" / "chroma"
    OUT_PATH     = REPO_ROOT / "data" / "processed" / "retrieval_results.json"

print(f"QUERIES_PATH : {QUERIES_PATH}")
print(f"CHROMA_PATH  : {CHROMA_PATH}")
print(f"OUT_PATH     : {OUT_PATH}")

In [ ]:
# Cell 3 - Set up ChromaDB
import shutil
import chromadb

if ON_KAGGLE:
    # Copy from read-only Kaggle input to writable working directory
    if CHROMA_PATH.exists():
        shutil.rmtree(CHROMA_PATH)
    shutil.copytree(CHROMA_SRC, CHROMA_PATH)
    print(f"Copied ChromaDB from Kaggle input to {CHROMA_PATH}")
else:
    # Download via kagglehub if not already present
    chroma_sqlite = CHROMA_PATH / "chroma.sqlite3"
    if chroma_sqlite.exists():
        print(f"ChromaDB already exists at {CHROMA_PATH} — skipping download.")
    else:
        import kagglehub
        dl_path = Path(kagglehub.dataset_download("rajlucka/fairsearch-qbio-chromadb"))
        # Locate chroma.sqlite3 inside the downloaded folder
        candidates = list(dl_path.rglob("chroma.sqlite3"))
        if not candidates:
            raise FileNotFoundError("chroma.sqlite3 not found in downloaded dataset.")
        src = candidates[0].parent
        CHROMA_PATH.mkdir(parents=True, exist_ok=True)
        for item in src.rglob("*"):
            if item.name == ".gitkeep":
                continue
            rel = item.relative_to(src)
            dst = CHROMA_PATH / rel
            if item.is_dir():
                dst.mkdir(parents=True, exist_ok=True)
            else:
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(item, dst)
        print(f"Downloaded and copied ChromaDB to {CHROMA_PATH}")

# Connect and verify
client     = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = client.get_collection("qbio_papers")
print(f"Collection 'qbio_papers': {collection.count():,} records")

In [ ]:
# Cell 4 - Load queries
import json

with open(QUERIES_PATH, "r") as f:
    queries = json.load(f)

print(f"Loaded {len(queries)} queries.")
print(f"Example: [{queries[0]['query_id']}] ({queries[0]['subcategory']}) {queries[0]['query_text'][:70]}")

In [ ]:
# Cell 5 - Pre-compute relevant pool size per subcategory (Recall@10 denominator)
# Fetches metadata in batches of 5000 to avoid SQLite "too many SQL variables" error

target_subcats = sorted(set(q["subcategory"] for q in queries))
print(f"Subcategories: {target_subcats}\n")

BATCH_SIZE = 5000
total = collection.count()
subcat_pool_size = {sc: 0 for sc in target_subcats}

print(f"Scanning {total:,} records in batches of {BATCH_SIZE}...")
for offset in range(0, total, BATCH_SIZE):
    batch = collection.get(limit=BATCH_SIZE, offset=offset, include=["metadatas"])
    for meta in batch["metadatas"]:
        cats = meta.get("categories", "")
        for sc in target_subcats:
            if sc in cats:
                subcat_pool_size[sc] += 1
    print(f"  processed {min(offset + BATCH_SIZE, total):,} / {total:,}", end="\r")

print("\nDone.\n")
for sc in sorted(subcat_pool_size):
    print(f"  {sc}: {subcat_pool_size[sc]:,} papers in corpus")
print(f"\nPool sizes computed for {len(subcat_pool_size)} subcategories.")

In [ ]:
# Cell 6 - Load embedding model (same model used in Step 2)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded: all-MiniLM-L6-v2")

test_vec = model.encode("test sentence")
print(f"Embedding dimension: {len(test_vec)}")

In [ ]:
# Cell 7 - Retrieve Top-10 for all 50 queries and compute metrics
from tqdm import tqdm

K = 10
results_records = []

for q in tqdm(queries, desc="Retrieving"):
    qid       = q["query_id"]
    qtext     = q["query_text"]
    subcat    = q["subcategory"]
    pool_size = max(subcat_pool_size.get(subcat, 0), 1)

    q_vec = model.encode(qtext).tolist()

    chroma_res = collection.query(
        query_embeddings=[q_vec],
        n_results=K,
        include=["metadatas", "distances"]
    )

    paper_ids = chroma_res["ids"][0]
    metadatas = chroma_res["metadatas"][0]
    distances = chroma_res["distances"][0]

    qtype = q.get("type", "neutral")

    if qtype == "contradictory":
        # subcategory-match proxy does not apply to debate queries
        relevance      = [None] * len(paper_ids)
        precision_at_k = None
        recall_at_k    = None
        hit_rate_at_k  = None
    else:
        relevance = [
            1 if subcat in meta.get("categories", "") else 0
            for meta in metadatas
        ]
        num_relevant   = sum(relevance)
        precision_at_k = round(num_relevant / K, 4)
        recall_at_k    = round(num_relevant / pool_size, 6)
        hit_rate_at_k  = 1 if num_relevant > 0 else 0

    results_records.append({
        "query_id"            : qid,
        "query_text"          : qtext,
        "subcategory"         : subcat,
        "type"                : qtype,
        "retrieved_paper_ids" : paper_ids,
        "relevance"           : relevance,
        "precision_at_10"     : precision_at_k,
        "recall_at_10"        : recall_at_k,
        "hit_rate_at_10"      : hit_rate_at_k,
    })

print(f"\nDone. {len(results_records)} queries evaluated.")

In [ ]:
# Cell 8 - Save results
import json

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w") as f:
    json.dump(results_records, f, indent=2)

print(f"Saved {len(results_records)} records to: {OUT_PATH}")

In [ ]:
# Cell 9 - Overall summary
import statistics

p_vals  = [r["precision_at_10"] for r in results_records if r["precision_at_10"] is not None]
rc_vals = [r["recall_at_10"]    for r in results_records if r["recall_at_10"]    is not None]
hr_vals = [r["hit_rate_at_10"]  for r in results_records if r["hit_rate_at_10"]  is not None]

print("=" * 50)
print(f"OVERALL BASELINE METRICS (K=10, n={len(p_vals)} neutral queries)")
print("=" * 50)
print(f"  Mean Precision@10 : {statistics.mean(p_vals):.4f}")
print(f"  Mean Recall@10    : {statistics.mean(rc_vals):.6f}  (low — large relevant pool)")
print(f"  Mean HitRate@10   : {statistics.mean(hr_vals):.4f}")
print(f"  HitRate=1 queries : {sum(hr_vals)}/{len(hr_vals)}")

In [ ]:
# Cell 10 - Per-subcategory breakdown
from collections import defaultdict

subcat_metrics = defaultdict(lambda: {"precision": [], "recall": [], "hit_rate": []})

for r in results_records:
    if r["precision_at_10"] is None:   # skip contradictory (no relevance proxy)
        continue
    sc = r["subcategory"]
    subcat_metrics[sc]["precision"].append(r["precision_at_10"])
    subcat_metrics[sc]["recall"].append(r["recall_at_10"])
    subcat_metrics[sc]["hit_rate"].append(r["hit_rate_at_10"])

print(f"{'Subcategory':<14} {'Queries':>7} {'Pool':>7} {'Prec@10':>9} {'Rec@10':>10} {'Hit@10':>8}")
print("-" * 60)

for sc in sorted(subcat_metrics):
    m      = subcat_metrics[sc]
    n      = len(m["precision"])
    avg_p  = statistics.mean(m["precision"])
    avg_r  = statistics.mean(m["recall"])
    avg_hr = statistics.mean(m["hit_rate"])
    pool   = subcat_pool_size.get(sc, 0)
    print(f"{sc:<14} {n:>7} {pool:>7,} {avg_p:>9.4f} {avg_r:>10.6f} {avg_hr:>8.4f}")

In [ ]:
# Cell 11 - Per-query detail (sorted by Precision@10 descending)
neutral_results = [r for r in results_records if r["precision_at_10"] is not None]
sorted_results  = sorted(neutral_results, key=lambda r: r["precision_at_10"], reverse=True)

print(f"{'QID':<6} {'Subcat':<12} {'Prec@10':>9} {'Hit@10':>8}  Query (truncated)")
print("-" * 90)
for r in sorted_results:
    print(
        f"{r['query_id']:<6} "
        f"{r['subcategory']:<12} "
        f"{r['precision_at_10']:>9.4f} "
        f"{r['hit_rate_at_10']:>8}  "
        f"{r['query_text'][:55]}"
    )

In [ ]:
# Cell 12 - Spot check: inspect Top-10 results for a specific query
INSPECT_QID = "q051"   # change to any query_id (v2 draft is q051-q150)

target   = next(r for r in results_records if r["query_id"] == INSPECT_QID)
q_subcat = target["subcategory"]

print(f"Query      : {target['query_text']}")
print(f"Subcategory: {q_subcat}")
print(f"Precision@10: {target['precision_at_10']}   HitRate@10: {target['hit_rate_at_10']}")
print()

q_vec = model.encode(target["query_text"]).tolist()
chroma_res = collection.query(
    query_embeddings=[q_vec],
    n_results=K,
    include=["metadatas", "distances"]
)

print(f"{'#':<3} {'Rel':<4} {'Dist':>6}  {'Categories':<35} Title")
print("-" * 100)
for rank, (pid, meta, dist, rel) in enumerate(
    zip(
        chroma_res["ids"][0],
        chroma_res["metadatas"][0],
        chroma_res["distances"][0],
        target["relevance"]
    ), start=1
):
    marker = "v" if rel else " "
    cats   = meta.get("categories", "")[:34]
    title  = meta.get("title", "")[:55]
    print(f"{rank:<3} [{marker}]  {dist:>6.4f}  {cats:<35} {title}")

In [ ]:
# Cell 13 - Year distribution: retrieved papers vs corpus (Preliminary Fairness Observations)
from collections import Counter

# --- A: collect years for all retrieved paper IDs ---
all_retrieved_ids = []
for r in results_records:
    all_retrieved_ids.extend(r["retrieved_paper_ids"])

id_counts   = Counter(all_retrieved_ids)   # how many queries retrieved each paper
unique_ids  = list(id_counts.keys())

retrieved_year_counter = Counter()
for i in range(0, len(unique_ids), 500):
    batch_res = collection.get(ids=unique_ids[i:i + 500], include=["metadatas"])
    for pid, meta in zip(batch_res["ids"], batch_res["metadatas"]):
        yr = meta.get("year")
        if yr is not None:
            retrieved_year_counter[int(yr)] += id_counts[pid]

# --- B: corpus-wide year distribution (reuse BATCH_SIZE=5000 from Cell 5) ---
print("Scanning corpus for year distribution...")
corpus_year_counter = Counter()
for offset in range(0, collection.count(), BATCH_SIZE):
    batch = collection.get(limit=BATCH_SIZE, offset=offset, include=["metadatas"])
    for meta in batch["metadatas"]:
        yr = meta.get("year")
        if yr is not None:
            corpus_year_counter[int(yr)] += 1
    print(f"  {min(offset + BATCH_SIZE, collection.count()):,} / {collection.count():,}", end="\r")
print("\nDone.\n")

# --- C: side-by-side comparison ---
total_retrieved = sum(retrieved_year_counter.values())   # 500 (50 queries × K=10)
total_corpus    = sum(corpus_year_counter.values())

all_years = sorted(set(retrieved_year_counter) | set(corpus_year_counter))

print(f"{'Year':>6} {'Corpus':>8} {'Corpus%':>9} {'Retr.':>7} {'Retr.%':>8}  {'Ratio':>7}")
print("-" * 58)
for yr in all_years:
    c = corpus_year_counter.get(yr, 0)
    r = retrieved_year_counter.get(yr, 0)
    c_pct = c / total_corpus * 100
    r_pct = r / total_retrieved * 100
    ratio = r_pct / c_pct if c_pct > 0 else float("inf")
    flag  = "  ▲▲" if ratio > 2.0 else ("  ▼▼" if (c_pct > 0.5 and ratio < 0.5) else "")
    print(f"{yr:>6} {c:>8,} {c_pct:>8.1f}% {r:>7,} {r_pct:>7.1f}%  {ratio:>6.2f}x{flag}")

# --- D: summary observation ---
print("\n=== Preliminary Observation: Year Bias ===")
ratios = [
    (yr, retrieved_year_counter.get(yr, 0) / total_retrieved,
         corpus_year_counter.get(yr, 0) / total_corpus)
    for yr in all_years if corpus_year_counter.get(yr, 0) > 0
]
by_ratio = sorted(ratios, key=lambda x: x[1] / x[2] if x[2] > 0 else 0, reverse=True)

print("Top 5 over-retrieved years (retrieved% / corpus%):")
for yr, r_f, c_f in by_ratio[:5]:
    print(f"  {yr}: {r_f*100:.1f}% retrieved vs {c_f*100:.1f}% corpus  ({r_f/c_f:.2f}x)")

print("\nTop 5 under-retrieved years:")
for yr, r_f, c_f in by_ratio[-5:]:
    ratio_val = r_f / c_f if c_f > 0 else 0
    print(f"  {yr}: {r_f*100:.1f}% retrieved vs {c_f*100:.1f}% corpus  ({ratio_val:.2f}x)")